# Pitcher P4 or Not Models — Train V2

P4 classification within D1 pitchers. Trained on D1 pitchers only (P4 + Non-P4 D1).

Uses 11 production features (all pitch metrics) plus `d1_prob` as a meta-feature (12 total).

**Source:** `backend/ml/train/train_v2/pitchers/pitchers_cleaned.csv`

In [8]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

PITCHER_CSV = Path("pitchers_cleaned.csv")

pitchers = pd.read_csv(PITCHER_CSV, low_memory=False)
print(f"Loaded {len(pitchers):,} rows, {len(pitchers.columns)} columns")
pitchers.head()

Loaded 19,506 rows, 44 columns


,name,link,player_state,high_school,class,resolved_position,positions,commitment,commitment_date,height,...,thirty_time_date,ten_yard_time,ten_yard_time_date,run_speed_max,run_speed_max_date,player_region,conference,division,college_location,committment_group
0,Enmanuel Acevedo,https://www.prepbaseballreport.com/profiles/NY...,NY,Poly Prep,2027,RHP,RHP,Virginia,11/12/25,76.0,...,NaN,NaN,NaN,NaN,NaN,Northeast,Atlantic Coast Conference,NCAA I,"CHARLOTTESVILLE, VA",P4
1,Riley Ackerman,https://www.prepbaseballreport.com/profiles/IN...,IN,Crown Point,2027,LHP,LHP,Northwestern,10/21/25,75.0,...,NaN,NaN,NaN,NaN,NaN,Midwest,BIG 10,NCAA I,"EVANSTON, IL",Non P4 D1
2,Connor Ackerman,https://www.prepbaseballreport.com/profiles/NY...,NY,St. Dominic High School,2027,RHP,"RHP,1B",Hofstra,9/22/25,76.0,...,6/26/25,1.78,6/26/25,18.0,6/26/25,Northeast,Colonial,NCAA I,"HEMPSTEAD, NY",Non P4 D1
3,Jace Adams,https://www.prepbaseballreport.com/profiles/AZ...,AZ,Flagstaff,2027,RHP,"RHP,3B",Arizona State (8/03/25),NaN,75.0,...,NaN,NaN,NaN,NaN,NaN,West,Big 12,NCAA I,"TEMPE, AZ",P4
4,Noah Adkins,https://www.prepbaseballreport.com/profiles/FL...,FL,Hagerty,2027,RHP,RHP,Stetson,10/02/25,73.0,...,NaN,NaN,NaN,NaN,NaN,South,Atlantic Sun,NCAA I,"DELAND, FL",Non P4 D1


In [9]:
P_MODELING_COLS = [
    "player_state", "resolved_position",
    "height", "weight", "throwing_hand", "hitting_handedness",
    "fastball_velo_max", "fastball_velo_max_date",
    "fastball_velo_range", "fastball_velo_range_date",
    "fastball_spin", "fastball_spin_date",
    "changeup_velo_range", "changeup_velo_range_date",
    "changeup_spin", "changeup_spin_date",
    "curveball_velo_range", "curveball_velo_range_date",
    "curveball_spin", "curveball_spin_date",
    "slider_velo_range", "slider_velo_range_date",
    "slider_spin", "slider_spin_date",
    "sixty_time", "sixty_time_date",
    "thirty_time", "thirty_time_date",
    "ten_yard_time", "ten_yard_time_date",
    "run_speed_max", "run_speed_max_date",
    "player_region", "committment_group", "commitment_date"
]

P_MODELING_COLS = [c for c in P_MODELING_COLS if c in pitchers.columns]
p_modeling = pitchers[P_MODELING_COLS]

# Filter to D1 only (P4 + Non P4 D1)
p_modeling = p_modeling[p_modeling["committment_group"] != "Unknown"]
p_modeling = p_modeling[p_modeling["committment_group"] != "Non D1"]

print(p_modeling["committment_group"].value_counts())
print(f"Shape: {p_modeling.shape}")

committment_group
Non P4 D1    4772
P4           1469
Name: count, dtype: int64
Shape: (6241, 35)


In [10]:
p_modeling["p4_or_not"] = (p_modeling["committment_group"] != "Non P4 D1").astype(int)

print(f"Target distribution:")
print(p_modeling["p4_or_not"].value_counts())
print(f"P4 rate: {p_modeling['p4_or_not'].mean():.1%}")

Target distribution:
p4_or_not
0    4772
1    1469
Name: count, dtype: int64
P4 rate: 23.5%


In [11]:
# ============================================================
# STALE DATA ANALYSIS
# ============================================================
STALE_MONTHS = 24
OUTLIER_STD_DEV = 2

p_model_recent = p_modeling.copy()
p_model_recent["class"] = pitchers.loc[p_model_recent.index, "class"]

STAT_DATE_PAIRS = [
    ("fastball_velo_max",    "fastball_velo_max_date"),
    ("fastball_velo_range",  "fastball_velo_range_date"),
    ("fastball_spin",        "fastball_spin_date"),
    ("changeup_velo_range",  "changeup_velo_range_date"),
    ("changeup_spin",        "changeup_spin_date"),
    ("curveball_velo_range", "curveball_velo_range_date"),
    ("curveball_spin",       "curveball_spin_date"),
    ("slider_velo_range",    "slider_velo_range_date"),
    ("slider_spin",          "slider_spin_date"),
    ("sixty_time",           "sixty_time_date"),
    ("thirty_time",          "thirty_time_date"),
    ("ten_yard_time",        "ten_yard_time_date"),
    ("run_speed_max",        "run_speed_max_date"),
]
STAT_DATE_PAIRS = [(s, d) for s, d in STAT_DATE_PAIRS if s in p_model_recent.columns and d in p_model_recent.columns]

def parse_pbr_date(d):
    if pd.isna(d) or str(d).strip() == "":
        return pd.NaT
    try:
        return pd.to_datetime(str(d).strip(), format="%m/%d/%y")
    except Exception:
        try:
            return pd.to_datetime(str(d).strip())
        except Exception:
            return pd.NaT

p_model_recent["grad_date"] = pd.to_datetime(p_model_recent["class"].astype(str) + "-06-01")

for stat_col, date_col in STAT_DATE_PAIRS:
    parsed_col = f"_{date_col}_parsed"
    months_col = f"{stat_col}__months_before_grad"
    p_model_recent[parsed_col] = p_model_recent[date_col].apply(parse_pbr_date)
    p_model_recent[months_col] = ((p_model_recent["grad_date"] - p_model_recent[parsed_col]).dt.days / 30.44).round(1)

print(f"Parsed {len(STAT_DATE_PAIRS)} stat date columns.\n")

print(f"{'='*70}")
print(f"STALENESS BY COMMITMENT GROUP  (threshold = {STALE_MONTHS} months)")
print(f"{'='*70}\n")

for group in ["P4", "Non P4 D1"]:
    mask = p_model_recent["committment_group"] == group
    g_stale = 0
    g_total = 0
    for stat_col, _ in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        has_val = p_model_recent.loc[mask, stat_col].notna()
        is_stale = p_model_recent.loc[mask, months_col] > STALE_MONTHS
        g_stale += (has_val & is_stale).sum()
        g_total += has_val.sum()
    pct = 100 * g_stale / g_total if g_total else 0
    n_players = mask.sum()
    print(f"  {group:>12}  ({n_players:>5,} players): {g_stale:>4,} / {g_total:>5,} values stale ({pct:.1f}%)")

stat_value_cols = [s for s, _ in STAT_DATE_PAIRS]
print(f"\n{'='*70}")
print(f"OUTLIER SUMMARY  (+-{OUTLIER_STD_DEV} std dev from group mean)")
print(f"{'='*70}\n")

for group in ["P4", "Non P4 D1"]:
    mask = p_model_recent["committment_group"] == group
    tot_outliers = 0
    tot_stale_o = 0
    for stat_col in stat_value_cols:
        months_col = f"{stat_col}__months_before_grad"
        vals = p_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std = vals.std()
        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue
        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        is_stale = p_model_recent.loc[mask, months_col] > STALE_MONTHS
        tot_outliers += is_outlier.sum()
        tot_stale_o += (is_outlier & is_stale).sum()
    tot_fresh_o = tot_outliers - tot_stale_o
    pct = 100 * tot_stale_o / tot_outliers if tot_outliers else 0
    print(f"  {group:>12}: {tot_outliers:>4} outliers -> {tot_stale_o:>3} stale ({pct:.0f}%), {tot_fresh_o:>3} fresh ({100-pct:.0f}%)")

internal_cols = [c for c in p_model_recent.columns if c.startswith("_")]
p_model_recent.drop(columns=internal_cols, inplace=True)
print(f"\np_model_recent shape: {p_model_recent.shape}")

Parsed 13 stat date columns.

STALENESS BY COMMITMENT GROUP  (threshold = 24 months)

            P4  (1,469 players): 3,002 / 10,804 values stale (27.8%)
     Non P4 D1  (4,772 players): 8,199 / 34,619 values stale (23.7%)

OUTLIER SUMMARY  (+-2 std dev from group mean)

            P4:  502 outliers -> 259 stale (52%), 243 fresh (48%)
     Non P4 D1: 1636 outliers -> 782 stale (48%), 854 fresh (52%)

p_model_recent shape: (6241, 51)


In [12]:
# ============================================================
# APPLY OPTION B — NaN only stale outliers (MULTI-PASS)
# ============================================================
MAX_PASSES = 5
grand_total = 0

for pass_i in range(1, MAX_PASSES + 1):
    pass_total = 0
    pass_log = []

    for stat_col, date_col in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        stat_nand = 0

        for group in ["P4", "Non P4 D1"]:
            mask = p_model_recent["committment_group"] == group
            vals = p_model_recent.loc[mask, stat_col]
            mean = vals.mean()
            std = vals.std()

            if pd.isna(mean) or pd.isna(std) or std == 0:
                continue

            is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
            is_stale = p_model_recent.loc[mask, months_col] > STALE_MONTHS

            to_nan = is_outlier & is_stale
            n = to_nan.sum()

            if n > 0:
                p_model_recent.loc[to_nan[to_nan].index, stat_col] = np.nan
                p_model_recent.loc[to_nan[to_nan].index, date_col] = np.nan
                stat_nand += n

        if stat_nand > 0:
            pass_log.append({"stat": stat_col, "values_removed": stat_nand})
            pass_total += stat_nand

    grand_total += pass_total
    print(f"Pass {pass_i}: NaN'd {pass_total} stale outlier values")
    if pass_log:
        for row in pass_log:
            print(f"    {row['stat']:>25}: {row['values_removed']}")

    if pass_total == 0:
        print(f"  No more stale outliers found — stopping.")
        break

print(f"\nTotal across {pass_i} passes: {grand_total} values NaN'd")

remaining = 0
for stat_col, _ in STAT_DATE_PAIRS:
    months_col = f"{stat_col}__months_before_grad"
    for group in ["P4", "Non P4 D1"]:
        mask = p_model_recent["committment_group"] == group
        vals = p_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std = vals.std()
        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue
        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        is_stale = p_model_recent.loc[mask, months_col] > STALE_MONTHS
        remaining += (is_outlier & is_stale).sum()

print(f"Stale outliers remaining: {remaining}")
print(f"p_model_recent shape: {p_model_recent.shape}")

Pass 1: NaN'd 1041 stale outlier values
            fastball_velo_max: 126
          fastball_velo_range: 118
                fastball_spin: 64
          changeup_velo_range: 142
                changeup_spin: 46
         curveball_velo_range: 158
               curveball_spin: 64
            slider_velo_range: 87
                  slider_spin: 26
                   sixty_time: 109
                  thirty_time: 31
                ten_yard_time: 35
                run_speed_max: 35
Pass 2: NaN'd 342 stale outlier values
            fastball_velo_max: 33
          fastball_velo_range: 35
                fastball_spin: 9
          changeup_velo_range: 53
                changeup_spin: 7
         curveball_velo_range: 49
               curveball_spin: 9
            slider_velo_range: 21
                  slider_spin: 7
                   sixty_time: 82
                  thirty_time: 12
                ten_yard_time: 8
                run_speed_max: 17
Pass 3: NaN'd 78 stale outlier values

## Logistic Regression Baseline

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

FEATURES = [
    "height", "weight",
    "fastball_velo_max", "fastball_velo_range", "fastball_spin",
    "changeup_velo_range", "changeup_spin",
    "curveball_velo_range", "curveball_spin",
    "slider_velo_range", "slider_spin",
]

X = p_model_recent[FEATURES]
y = p_model_recent["p4_or_not"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ("impute", KNNImputer(n_neighbors=10)),
    ("scale", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print("Logistic Regression Baseline — 11 Features")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=["Non P4 D1", "P4"]))
print(confusion_matrix(y_test, y_pred), "\n")

coefs = pd.Series(pipe["lr"].coef_[0], index=FEATURES).sort_values()
print("Feature coefficients (standardized):")
print(coefs)

Logistic Regression Baseline — 11 Features
              precision    recall  f1-score   support

   Non P4 D1       0.87      0.65      0.75       955
          P4       0.38      0.67      0.48       294

    accuracy                           0.66      1249
   macro avg       0.62      0.66      0.61      1249
weighted avg       0.75      0.66      0.68      1249

[[625 330]
 [ 96 198]] 

Feature coefficients (standardized):
fastball_velo_range    -0.072904
weight                 -0.056445
fastball_spin          -0.042168
changeup_velo_range    -0.019720
changeup_spin          -0.013829
curveball_velo_range    0.012682
slider_spin             0.056817
curveball_spin          0.072801
slider_velo_range       0.131090
height                  0.332588
fastball_velo_max       0.831238
dtype: float64


## Generate Out-of-Fold D1 Probabilities

We need `d1_prob` as a meta-feature for the P4 model. Can't use the saved D1 model directly
because these D1 players were in its training data — that would leak. Instead, retrain the D1
model on the **full pitcher dataset** (including Non-D1) with cross_val_predict to get leak-free
calibrated probabilities for every player, then filter to just the D1 subset.

In [15]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.calibration import CalibratedClassifierCV
import numpy as np

# Load the FULL pitcher dataset (including Non-D1) for D1 model training
pitchers_full = pd.read_csv(Path("pitchers_cleaned.csv"), low_memory=False)
p_full = pitchers_full.copy()
p_full = p_full[p_full["committment_group"] != "Unknown"]
p_full["d1_or_not"] = (p_full["committment_group"] != "Non D1").astype(int)

FEATURES_D1 = [
    "height", "weight",
    "fastball_velo_max", "fastball_velo_range", "fastball_spin",
    "changeup_velo_range", "changeup_spin",
    "curveball_velo_range", "curveball_spin",
    "slider_velo_range", "slider_spin",
]

X_full = p_full[FEATURES_D1]
y_full = p_full["d1_or_not"]
neg, pos = (y_full == 0).sum(), (y_full == 1).sum()

print(f"Full pitcher dataset: {len(p_full)} players ({pos} D1, {neg} Non-D1)")

xgb_d1 = XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.01,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=3.0, min_child_weight=5,
    scale_pos_weight=neg / pos, eval_metric="logloss",
    random_state=42, enable_categorical=False,
)
cal_d1 = CalibratedClassifierCV(xgb_d1, method="isotonic", cv=3)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = cross_val_predict(cal_d1, X_full, y_full, cv=skf, method="predict_proba")[:, 1]

p_full["d1_prob"] = oof_probs

# Filter to D1 players only and merge into p_model_recent
d1_players = p_full[p_full["d1_or_not"] == 1][["d1_prob"]]
p_model_recent = p_model_recent.join(d1_players, how="left")

print(f"\nD1 prob stats for D1 players (n={p_model_recent['d1_prob'].notna().sum()}):")
print(f"  mean: {p_model_recent['d1_prob'].mean():.3f}")
print(f"  std:  {p_model_recent['d1_prob'].std():.3f}")
print(f"  min:  {p_model_recent['d1_prob'].min():.3f}")
print(f"  max:  {p_model_recent['d1_prob'].max():.3f}")

print(f"\nD1 prob by group:")
for group in ["P4", "Non P4 D1"]:
    mask = p_model_recent["committment_group"] == group
    vals = p_model_recent.loc[mask, "d1_prob"]
    print(f"  {group:>12}: mean={vals.mean():.3f}  std={vals.std():.3f}  median={vals.median():.3f}")

Full pitcher dataset: 18183 players (6241 D1, 11942 Non-D1)

D1 prob stats for D1 players (n=6241):
  mean: 0.526
  std:  0.230
  min:  0.003
  max:  1.000

D1 prob by group:
            P4: mean=0.658  std=0.206  median=0.712
     Non P4 D1: mean=0.485  std=0.222  median=0.463


## XGBoost — P4 vs Non-P4 D1 (11 core + d1_prob)

Using all 11 production pitcher features plus the D1 probability as a meta-feature.
Heavier regularization than the D1 model due to smaller P4 sample size.

In [16]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

FEATURES_P4 = [
    "height", "weight",
    "fastball_velo_max", "fastball_velo_range", "fastball_spin",
    "changeup_velo_range", "changeup_spin",
    "curveball_velo_range", "curveball_spin",
    "slider_velo_range", "slider_spin",
    "d1_prob",
]

X = p_model_recent[FEATURES_P4]
y = p_model_recent["p4_or_not"]
neg, pos = (y == 0).sum(), (y == 1).sum()
print(f"Class balance: {neg} Non-P4 D1, {pos} P4  (ratio {neg/pos:.2f}:1)\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_p4 = XGBClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.5, reg_lambda=5.0, min_child_weight=8,
    scale_pos_weight=neg / pos, eval_metric="logloss",
    random_state=42, enable_categorical=False,
)

xgb_p4.fit(X_train, y_train)
y_pred = xgb_p4.predict(X_test)
y_proba = xgb_p4.predict_proba(X_test)[:, 1]

print(f"XGBoost — {len(FEATURES_P4)} Features (11 core + d1_prob)")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=["Non P4 D1", "P4"]))
print(confusion_matrix(y_test, y_pred))

auc = roc_auc_score(y_test, y_proba)
print(f"\nAUC-ROC: {auc:.4f}")

importances = pd.Series(xgb_p4.feature_importances_, index=FEATURES_P4).sort_values(ascending=False)
print(f"\nFeature importance:")
print(importances.to_string())

# Stratified 5-Fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv = cross_validate(
    xgb_p4, X, y, cv=skf,
    scoring=["roc_auc", "f1", "precision", "recall", "accuracy"],
    return_train_score=True,
)

print(f"\nStratified 5-Fold CV:")
print("=" * 60)
for metric in ["roc_auc", "accuracy", "f1", "precision", "recall"]:
    tr = cv[f"train_{metric}"]
    te = cv[f"test_{metric}"]
    label = "AUC-ROC" if metric == "roc_auc" else metric
    print(f"  {label:>12}:  train {tr.mean():.3f} (+/- {tr.std():.3f})  |  test {te.mean():.3f} (+/- {te.std():.3f})")
print(f"\n  Overfit gap (AUC): {cv['train_roc_auc'].mean() - cv['test_roc_auc'].mean():.3f}")

# Calibration
xgb_p4_cal = CalibratedClassifierCV(xgb_p4, method="isotonic", cv=5)
xgb_p4_cal.fit(X_train, y_train)
y_proba_cal = xgb_p4_cal.predict_proba(X_test)[:, 1]
y_pred_cal = (y_proba_cal >= 0.35).astype(int)

fraction_raw, mean_raw = calibration_curve(y_test, y_proba, n_bins=8)
fraction_cal, mean_cal = calibration_curve(y_test, y_proba_cal, n_bins=8)

brier_raw = brier_score_loss(y_test, y_proba)
brier_cal = brier_score_loss(y_test, y_proba_cal)

print(f"\nCalibrated model (threshold=0.35):")
print("=" * 60)
print(classification_report(y_test, y_pred_cal, target_names=["Non P4 D1", "P4"]))
print(confusion_matrix(y_test, y_pred_cal))

print(f"\nBrier:  raw = {brier_raw:.4f}  |  calibrated = {brier_cal:.4f}  |  improvement = {brier_raw - brier_cal:.4f}")

print(f"\nCalibration comparison (raw vs calibrated):")
print("=" * 75)
print(f"  {'--- RAW ---':^28}   |   {'--- CALIBRATED ---':^28}")
print(f"  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}   |   {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}")
print(f"  {'-'*28}   |   {'-'*28}")

max_rows = max(len(mean_raw), len(mean_cal))
for i in range(max_rows):
    if i < len(mean_raw):
        r_gap = mean_raw[i] - fraction_raw[i]
        raw_str = f"  {mean_raw[i]:>10.2f}  {fraction_raw[i]:>8.2f}  {r_gap:>+8.2f}"
    else:
        raw_str = f"  {'':>28}"
    if i < len(mean_cal):
        c_gap = mean_cal[i] - fraction_cal[i]
        cal_str = f"{mean_cal[i]:>10.2f}  {fraction_cal[i]:>8.2f}  {c_gap:>+8.2f}"
        flag = "  <- off" if abs(c_gap) > 0.10 else ""
        cal_str += flag
    else:
        cal_str = f"{'':>28}"
    print(f"{raw_str}   |   {cal_str}")

auc_cal = roc_auc_score(y_test, y_proba_cal)
max_gap_cal = max(abs(mean_cal[i] - fraction_cal[i]) for i in range(len(mean_cal)))
print(f"\nAUC-ROC (calibrated): {auc_cal:.4f}")
print(f"Max calibration gap: {max_gap_cal:.2f}")

Class balance: 4772 Non-P4 D1, 1469 P4  (ratio 3.25:1)

XGBoost — 12 Features (11 core + d1_prob)
              precision    recall  f1-score   support

   Non P4 D1       0.87      0.73      0.79       955
          P4       0.42      0.64      0.51       294

    accuracy                           0.71      1249
   macro avg       0.64      0.68      0.65      1249
weighted avg       0.76      0.71      0.73      1249

[[697 258]
 [107 187]]

AUC-ROC: 0.7450

Feature importance:
d1_prob                 0.300629
fastball_velo_max       0.182148
fastball_velo_range     0.078966
height                  0.073308
slider_velo_range       0.053510
curveball_spin          0.048128
weight                  0.045241
changeup_velo_range     0.044931
curveball_velo_range    0.044311
fastball_spin           0.043404
changeup_spin           0.042765
slider_spin             0.042657

Stratified 5-Fold CV:
       AUC-ROC:  train 0.803 (+/- 0.003)  |  test 0.735 (+/- 0.012)
      accuracy:  train 0.73

In [17]:
import joblib, json, os
from datetime import datetime

# ============================================================
# SAVE CALIBRATED P4 MODEL — PITCHERS
# ============================================================
VERSION = f"version_{datetime.now().strftime('%m%d%Y')}"
SAVE_DIR = os.path.abspath(os.path.join(
    os.getcwd(), "..", "..", "..", "models", "models_p", "models_p4_or_not_p", VERSION
))
os.makedirs(SAVE_DIR, exist_ok=True)

THRESHOLD = 0.35

# 1. Save calibrated model
model_path = os.path.join(SAVE_DIR, "calibrated_xgb_model.pkl")
joblib.dump(xgb_p4_cal, model_path)
print(f"Saved calibrated model -> {model_path}")

# 2. Model config
model_config = {
    "model_type": "XGBClassifier + CalibratedClassifierCV (isotonic)",
    "task": "P4 vs Non-P4 D1 — Pitchers",
    "version": VERSION,
    "created": datetime.now().isoformat(),
    "threshold": THRESHOLD,
    "n_features": len(FEATURES_P4),
    "features": FEATURES_P4,
    "hyperparameters": {
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 1.5,
        "reg_lambda": 5.0,
        "min_child_weight": 8,
        "scale_pos_weight": round(neg / pos, 4),
    },
    "calibration": {
        "method": "isotonic",
        "cv": 5,
    },
    "metrics": {
        "holdout_auc_roc": round(auc_cal, 4),
        "holdout_brier_raw": round(brier_raw, 4),
        "holdout_brier_calibrated": round(brier_cal, 4),
        "max_calibration_gap": round(max_gap_cal, 4),
        "cv_test_auc_mean": round(cv["test_roc_auc"].mean(), 4),
        "cv_test_auc_std": round(cv["test_roc_auc"].std(), 4),
        "cv_train_auc_mean": round(cv["train_roc_auc"].mean(), 4),
        "overfit_gap_auc": round(cv["train_roc_auc"].mean() - cv["test_roc_auc"].mean(), 4),
    },
    "calibration_curve": {
        "predicted": [round(float(x), 4) for x in mean_cal],
        "actual": [round(float(x), 4) for x in fraction_cal],
    },
    "training_data": {
        "total_d1_players": int(neg + pos),
        "p4_players": int(pos),
        "non_p4_d1_players": int(neg),
    },
}

config_path = os.path.join(SAVE_DIR, "model_config.json")
with open(config_path, "w") as f:
    json.dump(model_config, f, indent=2)
print(f"Saved model config -> {config_path}")

# 3. Feature metadata
feature_metadata = {
    "features": FEATURES_P4,
    "n_features": len(FEATURES_P4),
    "meta_features": {
        "d1_prob": {
            "source": "Out-of-fold predictions from pitcher D1 model (CalibratedClassifierCV, isotonic, cv=3)",
            "generation": "cross_val_predict with StratifiedKFold(n_splits=5) on full pitcher dataset (D1 + Non-D1)",
            "note": "Leak-free OOF probabilities — each player's d1_prob was predicted by a model that never saw that player in training",
        }
    },
    "feature_importance": {k: round(float(v), 4) for k, v in importances.items()},
    "position": "P",
    "core_features": [
        "height", "weight",
        "fastball_velo_max", "fastball_velo_range", "fastball_spin",
        "changeup_velo_range", "changeup_spin",
        "curveball_velo_range", "curveball_spin",
        "slider_velo_range", "slider_spin",
    ],
}

feat_path = os.path.join(SAVE_DIR, "feature_metadata.json")
with open(feat_path, "w") as f:
    json.dump(feature_metadata, f, indent=2)
print(f"Saved feature metadata -> {feat_path}")

# Summary
print(f"\n{'=' * 60}")
print(f"PITCHER P4 MODEL SAVED")
print(f"{'=' * 60}")
print(f"  Directory:    {SAVE_DIR}")
print(f"  Threshold:    {THRESHOLD}")
print(f"  Features:     {len(FEATURES_P4)} ({', '.join(FEATURES_P4)})")
print(f"  AUC-ROC:      {auc_cal:.4f}")
print(f"  Brier (cal):  {brier_cal:.4f}")
print(f"  Max cal gap:  {max_gap_cal:.2f}")
print(f"  Overfit gap:  {cv['train_roc_auc'].mean() - cv['test_roc_auc'].mean():.3f}")
print(f"\nFiles:")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:>30}  ({size:,} bytes)")

Saved calibrated model -> /Users/ryankolodziejczyk/Documents/AI Baseball Recruitment/code/backend/ml/models/models_p/models_p4_or_not_p/version_04212026/calibrated_xgb_model.pkl
Saved model config -> /Users/ryankolodziejczyk/Documents/AI Baseball Recruitment/code/backend/ml/models/models_p/models_p4_or_not_p/version_04212026/model_config.json
Saved feature metadata -> /Users/ryankolodziejczyk/Documents/AI Baseball Recruitment/code/backend/ml/models/models_p/models_p4_or_not_p/version_04212026/feature_metadata.json

PITCHER P4 MODEL SAVED
  Directory:    /Users/ryankolodziejczyk/Documents/AI Baseball Recruitment/code/backend/ml/models/models_p/models_p4_or_not_p/version_04212026
  Threshold:    0.35
  Features:     12 (height, weight, fastball_velo_max, fastball_velo_range, fastball_spin, changeup_velo_range, changeup_spin, curveball_velo_range, curveball_spin, slider_velo_range, slider_spin, d1_prob)
  AUC-ROC:      0.7468
  Brier (cal):  0.1531
  Max cal gap:  0.10
  Overfit gap:  0.0